In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# Dynamic programming: solving the Bellman equation with a model

## Where this session fits

> **The limitation we start from: the equation has the value on both sides, so it is not yet an answer**

- **The question:** can we solve the Bellman equation exactly?
- **Position: Part A (value-based methods), session 2**
    - the last session defined the value and the Bellman equation, in an average version and a max version
    - this session solves it exactly, by iterating, on a problem whose rules (the model) are known
- **Series goal:** choose good actions by trial and error, from small tables to robots and language models

**By the end you can**

1. run value iteration and policy iteration on a small grid
2. say how policy iteration moves the max out of the sweep
3. say what *evaluate* and *improve* are, and how both algorithms are loops of them
4. say what dynamic programming needs, and why that is its limitation


**Recall: the Bellman equation of the last session**

> **The value of a state is an average (for a policy $\pi$) or a max (for the best behavior) of one reward plus the discounted value of the next state.**

- the average version, for a policy $\pi$:
$$
\statevalfun_\pi(\state) = \sum_{\act} \pi(\act \mid \state) \sum_{\state', \rew} \transpm(\state', \rew \mid \state, \act)\bigl(\rew + \disc\,\statevalfun_\pi(\state')\bigr)
$$
- the max version, for the best behavior:
$$
\statevalfun_*(\state) = \max{\act} \sum_{\state', \rew} \transpm(\state', \rew \mid \state, \act)\bigl(\rew + \disc\,\statevalfun_*(\state')\bigr)
$$
- this session turns each equation into an update that is repeated

# Two dynamic-programming algorithms: value iteration and policy iteration

**Why this section?** The Bellman equation gives a recipe for computing values when the model is known. Two algorithms use it: value iteration and policy iteration.


> **This session has one theme: two dynamic-programming algorithms that solve the Bellman equation exactly when the model is known.**



**Terms used below**

1. **sweep:** one pass of the update over every state (the index $k$ counts sweeps)
2. **round:** one policy evaluation followed by one policy improvement (policy iteration)
    - the index $k$ counts rounds, and the sweeps inside a round use $j$
3. **greedy:** a policy is greedy with respect to a value if it picks the action with the best one-step lookahead

**The lake** is the one of the last session

- squares $0$ to $15$, row by row
- non-slippery, discount $0.9$

## Algorithm 1 of 2: value iteration

> **Update the value function until it stops changing, then read the policy off it by a one-step lookahead. The update is the one-step form (A1), computed exactly: with the model, it averages over all successors.**

- $\statevalfun_k$ is the table after $k$ sweeps: an *estimate*, not the value itself ($\statevalfun_0$ is all zeros)
    - it carries no policy subscript: this update looks for the best behavior
    - $k$ counts sweeps, not time steps (that is $\tt$)
- in sweep $k+1$, the value of *each* state $\state \in \States$ is updated via the Bellman update

$$
\statevalfun_{k+1}(\state) = \max{\act} \sum_{\state', \rew} \transpm(\state', \rew \mid \state, \act)\bigl(\rew + \disc \statevalfun_k(\state')\bigr)
$$

- $\rew$ is the reward the environment returns together with the successor state $\state'$ (the model gives the chance of each pair)
- the fixed point is the Bellman *optimality* equation
$$
\statevalfun_*(\state) = \max{\act} \sum_{\state', \rew} \transpm(\state', \rew \mid \state, \act)\bigl(\rew + \disc \statevalfun_*(\state')\bigr)
$$
- value iteration repeats the update until the table stops changing, then derives the policy
- vocabulary: the book says "iteration" for two things
    - a pass over the states: we say **sweep**
    - a whole evaluate-then-improve round: we say **round**


**How information moves: one step back per sweep**

> **Each sweep moves information one step back, from successors to predecessors, so $i$ sweeps carry it $i$ steps.**

- $\statevalfun_{k+1}(\state)$ uses the previous sweep's estimate $\statevalfun_k(\state')$ of each successor $\state'$, and chooses the action leading to the highest value
- on Frozen Lake the only non-zero reward is for entering G
    - in sweep $1$ it reaches only G's predecessor, square $14$
    - each further sweep reaches the predecessors one step farther from G
- so a state $i$ steps from the reward needs $i$ sweeps: hence we repeat until convergence


**Value iteration on Frozen Lake** (non-slippery, discount $0.9$)

<img src="images/value_iteration_strip.png" width=70%>

- sweep 1: only the square next to the goal gains value
- each further sweep moves information one square back from G: from successors back to predecessors
- after $6$ sweeps the values stop changing (sweep $7$ confirms that nothing changes): they are the optimal values, and the arrows are the optimal policy

**Value iteration on Frozen Lake: details**

- every square is updated from the *previous* sweep's table (synchronous)
- worked backup, square $10$ at sweep $2$: down reaches square $14$, worth $1.00$ after sweep $1$, so $\statevalfun_2(10) = 0 + 0.9 \times 1.00 = 0.90$ (right is a hole)
- the pseudo-code on the next slides overwrites $\statevalfun$ in place instead
- the result is the same


**Subtlety**

Prior to convergence
- $\statevalfun_k(\state)$ is a *non-random* variable
    - $\transpm(\state', \rew \mid \state, \act)$ is *known* for all $\state, \act$ since method is model-based
- the estimate is *not yet correct*
    - its error comes from stopping early, not from randomness

We will subsequently contrast this to the iterative estimates produced by model-free methods.


**Pseudo-code for value iteration** (the algorithm of Sutton and Barto, PDF page 105, in our notation; the book's textbook version is in an optional aside, not shown in the lecture)

- initialize $\statevalfun(\state) = 0$ for every state (terminal states keep $0$ and are never updated)
- repeat
    - $\Delta \leftarrow 0$
    - for each state $\state$
        - $v_{\text{old}} \leftarrow \statevalfun(\state)$
        - $\statevalfun(\state) \leftarrow \max{\act} \sum_{\state', \rew} \transpm(\state', \rew \mid \state, \act)\bigl(\rew + \disc\,\statevalfun(\state')\bigr)$
        - $\Delta \leftarrow \max{}\bigl(\Delta, \, |v_{\text{old}} - \statevalfun(\state)|\bigr)$
    - until $\Delta$ is below a small threshold
- then, for each state: $\pi(\state) \leftarrow \argmax{\act} \sum_{\state', \rew} \transpm(\state', \rew \mid \state, \act)\bigl(\rew + \disc\,\statevalfun(\state')\bigr)$
- return $\pi$ and $\statevalfun$


**Aside: the textbook's version of value iteration** (for your own review, in the book's notation)

<br>

<table>
    <center><strong>Value iteration</strong></center>
    <tr>
        <img src="images/RL_value_iteration_alg.png" width=90%>
    </tr>
    

</table>

Attribution: http://incompleteideas.net/book/RLbook2020.pdf#page=105

## Algorithm 2 of 2: policy iteration

**Why a second method?** Value iteration works, and it keeps no explicit policy until the end. Policy iteration organizes the same computation differently.

> **Policy iteration improves $\pi_k$ to $\pi_{k+1}$ by alternating two steps in round $k$: evaluate the current policy, then make it greedy.**

- **policy evaluation:** compute $\statevalfun_{\pi_k}$, the value of the *current* policy $\pi_k$
- **policy improvement:** make $\pi_{k+1}$ greedy with respect to $\statevalfun_{\pi_k}$
- policy iteration keeps an **explicit policy** at every round, and stops exactly after finitely many rounds (there are finitely many deterministic policies)
- whether it does less work depends on the problem: on the small lake, $27$ against $28$ action-backups per state (a later slide)


**What changes: the max over $\act$ becomes the policy's action $\pi(\state)$**

Value iteration: the action is *searched* for, by the max
$$
\statevalfun_{k+1}(\state) = \max{\act} \sum_{\state', \rew} \transpm(\state', \rew \mid \state, \act)\bigl(\rew + \disc \statevalfun_k(\state')\bigr)
$$

Policy evaluation: the action is *given*, by the current policy
$$
\statevalfun_{j+1}(\state) = \sum_{\state', \rew} \transpm\bigl(\state', \rew \mid \state, \boxed{\pi(\state)}\bigr)\Bigl(\rew + \disc \statevalfun_j(\state')\Bigr)
$$

- the only change is in the boxes: the $\act$ that the max searched over is replaced by $\pi(\state)$
    - the sweep index is $j$ here, because $k$ counts the rounds of policy iteration
    - one fixed action per state
    - so there is no search inside the sweep
- in this session the policy is deterministic: $\pi(\state)$ is the one action it chooses in $\state$ 
    - earlier sessions wrote $\pi(\act \mid \state)$, a probability; A3 and A4 return to that form
- the search happens once per round, in policy improvement
$$
\pi_{k+1}(\state) = \argmax{\act} \sum_{\state', \rew} \transpm(\state', \rew \mid \state, \act)\bigl(\rew + \disc \statevalfun_{\pi_k}(\state')\bigr)
$$


**Pseudo-code for policy iteration** (the algorithm of Sutton and Barto, PDF page 102, in our notation; the book's textbook version is in an optional aside, not shown in the lecture)

- initialize a policy $\pi$ arbitrarily (an arbitrary action in each state) and $\statevalfun(\state) = 0$ for every state (terminal states keep $0$)
- repeat **(outer loop: one round per pass; the policy changes only here)**
    - **policy evaluation**: compute $\statevalfun$ for the current $\pi$
        - **inner loop: one sweep per pass, $\pi$ held fixed**
        - repeat, for each state $\state$: $\statevalfun(\state) \leftarrow \sum_{\state', \rew} \transpm(\state', \rew \mid \state, \pi(\state))\bigl(\rew + \disc\,\statevalfun(\state')\bigr)$
        - until the values change by less than a small threshold
    - **policy improvement:** policy-stable $\leftarrow$ true; for each state $\state$
        - $\act_{\text{old}} \leftarrow \pi(\state)$
        - $\pi(\state) \leftarrow \argmax{\act} \sum_{\state', \rew} \transpm(\state', \rew \mid \state, \act)\bigl(\rew + \disc\,\statevalfun(\state')\bigr)$ (on a tie keep $\act_{\text{old}}$)
        - if $\act_{\text{old}} \neq \pi(\state)$, policy-stable $\leftarrow$ false
    - until policy-stable
- return $\pi$ and $\statevalfun$


**Aside: the textbook's version of policy iteration** (for your own review, in the book's notation)

<br>

<table>
    <center><strong>Policy iteration</strong></center>
    <tr>
        <img src="images/RL_policy_iteration_alg.png" width=90%>
    </tr>
    

</table>

Attribution: http://incompleteideas.net/book/RLbook2020.pdf#page=102

**Policy iteration on Frozen Lake**

<img src="images/policy_iteration_rounds.png" width=62%>

- start from *always down*
- each round evaluates the current policy, then improves it greedily
- the policy is optimal after round $3$; round $4$ only confirms that nothing changes
- orange arrows mark the actions that changed

Both Value Iteration and Policy Iteration converge to an optimal policy (they may differ only where actions tie).

The benefit of alternating between Policy Evaluation and Policy Improvement
- fewer rounds to converge (each round costs more) 
    - the Value function under the current policy is fully evaluated
    - before the Policy is updated

On Frozen Lake (computed)
- value iteration needs $7$ sweeps (the last confirms that nothing changes)
- policy iteration from "always down" needs $4$ rounds, but $11$ evaluation sweeps in total
- fewer rounds does not mean less work


**Why both ALGORITHMS converge**

- why they converge
    - improving a policy never makes it worse
        - greedy with respect to the values of $\pi$ picks, in every state, an action at least as good as $\pi$'s own
    - a policy that no longer changes satisfies the optimality equation
    - with finitely many policies, policy iteration stops
- caution: reaching the answer in finitely many sweeps is a feature of this small, deterministic lake
    - the best route there is short and never revisits a square
- in general the values approach the optimum only in the limit


**Aside (technical; you can skip it): how fast, and how long**

> **Nothing later depends on this slide.**

- **value iteration:** each sweep shrinks the largest error by a factor of at most $\disc$
    - after $k$ sweeps the error is at most $\disc^k$ times the starting error
    - this is geometric: fast for small $\disc$, slow when $\disc$ is close to $1$
- **policy iteration:** each round gives a policy at least as good as the one before
    - there are finitely many deterministic policies, so it stops after finitely many rounds
    - it stops at an exactly optimal policy
    - in practice it needs few rounds, but each round evaluates a policy fully


# Interlude: evaluate versus improve

**Why this section?** Both algorithms did two things, in different orders. Naming them now gives a loop that returns in every later method.

1. **evaluate:** compute the value of a given policy (iterate the policy's Bellman equation)
2. **improve:** make the policy greedy with respect to a value (the one-step lookahead of the last session)

<img src="images/gpi_loop.png" width=66%>

- the two push each other
    - a better value makes a better greedy policy
    - a better policy has a different value
    - they stop when neither changes: the optimum $\statevalfun_*$ and $\pi_*$

**The two algorithms of this session are two ways of running the evaluate-improve loop**

> **The two algorithms differ in how much evaluation they do before each improvement.**
>
> - **Policy iteration:** evaluate *fully*, then improve, and repeat
> - **Value iteration:** improve, then a *single* sweep of evaluation (the two steps merged into one update with a max)

- this loop is called *generalized policy iteration*
- later methods run the same loop with samples


## Comparing the two algorithms: value iteration and policy iteration

**Why this section?** A side-by-side of the two algorithms, as two ways of running the loop.

> **Both algorithms sweep every state. Inside a backup, value iteration takes $\max{\act}$ over all actions. Policy evaluation uses only the one action $\pi(\state)$.**

- so a policy-evaluation backup examines $1$ action per state, a value-iteration backup examines all of them
- policy iteration still looks at all actions, but only in the **improve** step, once per round
    - small lake: value iteration $\approx 7 \times 4 = 28$ action-backups per state
    - policy iteration $\approx 11 \times 1 + 4 \times 4 = 27$
    - the saving comes from searching all actions rarely, not every sweep
- **policy iteration** changes the policy only in the outer loop (improve)
    - during each evaluation the policy is **held fixed**, which restricts the backup to the action $\pi(\state)$
- value iteration stops when the values stop changing
- policy iteration stops when the policy stops improving


## The limitation: dynamic programming needs the model

**Why this section?** Everything above used $\transpm(\state', \rew \mid \state, \act)$. Where do they come from?

- every update **sums over all successor states, using the known model**
    - a board game's rules, or a grid we have written down, provide it
    - a robot, a market or a conversation does not
- every sweep also visits **every state**: the cost grows with the size of the state space
- so dynamic programming is exact and simple, but it needs the model and a small state space
- the next session removes the first requirement: learn the values from *samples* of experience instead


# What we established, and what it sets up

- **Established: two dynamic-programming algorithms solve the Bellman equation exactly, given the model**
    - **value iteration** repeats the Bellman optimality update (with a max over actions) until the values stop changing
    - **policy iteration** alternates evaluating the current policy and improving it
    - both are loops of **evaluate** and **improve**
    - the loop returns in later methods
    - both need the known model, and sweep every state
    - **model ledger:** $\transpm$ appears in every update here
        - it is the sum over successors
        - A3 removes it from value estimation
- **Sets up**
    - next session (A3): the model is unknown, so values are learned from sampled experience (Monte Carlo and temporal difference)
    - that raises the questions of bias and variance
    - watch the notation: here $\state$ and $\act$ are *any* state and action (the sweep visits all of them)
    - in A3 the capitals $\stateseq_\tt$, $\actseq_\tt$ are only the ones on one episode
    - later: **control**, learning to act and not only to evaluate (A4)
    - later: a network in place of the table when it is too big (A5)
